## Exploring OpenAI-Compatible APIs & MLflow

RDU's idea of AI-powered assistant for its students and professors start taking shape. Let's look at what is happening in the background before we go into more complex use cases.

In this notebook, we'll explore two key pieces of our current Canopy application:

- **OpenAI-compatible client** — to send prompts to our model
- **MLflow** — to manage prompts and trace interactions

Let's get hands on!

![](https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExNnRhZXg4eDlpZG83NnloeWJ6aGw3ZGp0bWl1azdobmNza2dobTNuYyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/uENJtJp1SGd597E7tL/giphy.gif)

## 1. Connect to the Model

As the developers of Canopy, we are given an access to a model. The model server exposes an **OpenAI-compatible API**, therefore we can use the standard `openai` Python library to talk to it.

We just need to point it to our model's endpoint.

In [ ]:
from openai import OpenAI

LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:80"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required", #for now
)

## 2. Your System Prompt

Bring your system prompt here, update the cell below and start experimenting:

In [ ]:
sys_prompt = """You are a helpful assistant. Summarize this:"""

#### Here is an example user message about Canopy. Let's summarize it!

In [ ]:
user_message = """
Canopy (Biology)

In biology and ecology, the canopy refers to the upper layer or \"roof\" formed by the crowns of trees in a forest or wooded area. This layer plays a critical role in regulating the ecosystem by controlling light penetration, humidity, temperature, and wind flow within the forest environment. The canopy is typically made up of the tallest trees and their branches and leaves, which often form a dense, continuous cover that can be several meters thick.

One of the primary ecological functions of the canopy is to provide habitat and food sources for a wide range of organisms. Many species of birds, insects, mammals, and epiphytes (plants that grow on other plants) are specially adapted to live in this elevated environment. The canopy also acts as a barrier that reduces the impact of heavy rain on the forest floor, helping to prevent soil erosion and maintain soil fertility.

Moreover, the canopy plays a crucial role in photosynthesis on a large scale by capturing sunlight and converting it into chemical energy, which sustains the forest's plant life and, consequently, the animals that depend on it. In tropical rainforests, the canopy is often so dense that very little sunlight reaches the forest floor, shaping the types of plants and animals that can survive in the understory and ground layers.

Scientists study canopies using specialized tools and methods such as canopy cranes, drones, and climbing equipment to better understand their structure, biodiversity, and ecological functions. This knowledge is vital for conservation efforts, particularly as canopies are sensitive to deforestation, climate change, and human activities that threaten their integrity.

Understanding the canopy's complexity helps ecologists appreciate the interdependent relationships within forests and the critical services these ecosystems provide, including carbon storage, oxygen production, and climate regulation. Protecting the canopy is essential to maintaining biodiversity and the health of our planet.

"""

## 3. Send a Request

Let's send a streaming chat completion request. This is the same API shape you'd use with OpenAI — the only difference is the `base_url` pointing to our self-hosted model.

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

Because our model server speaks the OpenAI protocol, you can swap models without changing your application code — just update the endpoint and model name. No model-specific logic needed!

---
## 4. Enter MLflow

Now let's bring **MLflow** into the picture. At the moment MLflow helps us:

- **Manage prompts** in a versioned registry so we don't lose track of what worked

First, let's configure the connection to our MLflow server.

In [ ]:
import os
import mlflow

MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        os.environ["MLFLOW_WORKSPACE"] = f.read().strip()

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("canopy-experiment")

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Workspace: {os.environ.get('MLFLOW_WORKSPACE', 'not set')}")

## 5. Prompt Registry

Remember the prompt you saved in the MLflow Prompt Registry earlier? Let's fetch it from here!

Update the `PROMPT_NAME` below with the name you used when creating your prompt in the registry. For example, if you called it `summarization`, use that.

In [ ]:
PROMPT_NAME = "summarization"  # 👈 Update this with YOUR prompt name!

In [ ]:
prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")

print("Loaded prompt from registry:")
print(prompt.template)

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": prompt.template},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

Now your prompt lives in the registry instead of being hardcoded. You can version it, tag it, and share it across applications — no code changes needed when you update it!

## 6. Save a New Prompt Version

You can also version prompts programmatically. Let's register a new version of your prompt with a different template and tag it.

In [ ]:
new_prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template="You are a concise academic assistant. Summarize the following text in 3 bullet points:",
    commit_message="Shorter bullet-point style summarization prompt",
    tags={"style": "bullet-points"},
)

mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="champion", version=new_prompt.version)

print(f"Registered prompt: {new_prompt.name}")
print(f"Version: {new_prompt.version}")
print(f"Alias: champion")
print(f"Template:\n{new_prompt.template}")

Head over to the OpenShift AI dashboard and check — you should see a new version of your prompt with its tags and a `champion` alias. Calling `register_prompt` with the same name creates a new version automatically!

![mlflow-champion-prompt.png](./mlflow-champion-prompt.png)

Now let's load the prompt by its alias and try it out:

In [ ]:
champion_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@champion")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": champion_prompt.template},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

---

Go back to the instructions to bring all of this to the Canopy backend, separating the LLM logic from the frontend!